음음 그래그래그건 사실이야

In [1]:
import torch.nn as nn
import torch


In [7]:
import tqdm as notebook_tqdm

In [8]:
from datasets import load_dataset

dataset = load_dataset("imdb")
train_dataset = dataset['train']
test_dataset = dataset['test']

from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


In [9]:

def tokenize_function(examples):
    tokenized = tokenizer(examples['text'], padding="max_length", truncation=True)
    
    # 특정 필드만 float로 변환 (예: 'input_ids')
    if 'input_ids' in tokenized:
        tokenized['input_ids'] = [torch.tensor(ids, dtype=torch.float) for ids in tokenized['input_ids']]
    
    return tokenized

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 50000/50000 [01:52<00:00, 443.46 examples/s]


In [63]:
from torch.optim import Adam
from torch.utils.data import DataLoader


# 데이터 준비
train_dataset = tokenized_datasets["train"].remove_columns(["text"])
train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format("torch")

test_dataset = tokenized_datasets["test"].remove_columns(["text"])
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format("torch")

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)
test_dataloader = DataLoader(test_dataset, batch_size=8)




In [18]:
train_dataset['input_ids'][0].shape

torch.Size([512])

In [20]:
input_ids.shape

torch.Size([8, 512])

### 모델 부분

In [60]:
class MyGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(MyGRUCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.x2h = nn.Linear(input_size, 3 * hidden_size, bias=True)
        self.h2h = nn.Linear(hidden_size, 3 * hidden_size, bias=True)

    def forward(self, x, h):
        x_gates = self.x2h(x)
        h_gates = self.h2h(h)
        
        x_r, x_z, x_n = x_gates.chunk(3, 1)
        h_r, h_z, h_n = h_gates.chunk(3, 1)

        r = torch.sigmoid(x_r + h_r)
        z = torch.sigmoid(x_z + h_z)
        n = torch.tanh(x_n + r * h_n)
        h_new = (1 - z) * n + z * h
        return h_new

In [61]:
class MyModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MyModel, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()

        self.gru_cell = MyGRUCell(input_size=input_size, hidden_size=hidden_size)

    def forward(self, x, h=None):
        if h is None:
            h = torch.zeros(x.size(0), self.hidden_size, device=x.device, dtype=torch.float)
        outputs = []
        for i in range(x.size(1)):
            h = self.gru_cell(x[:, i, :], h)
            outputs.append(h)
        
        output = outputs[-1]  # 마지막 hidden state 사용
        output = self.fc(output)
        output = self.sigmoid(output)
        return output, h

In [62]:
# 학습 루프
# 모델 인스턴스 생성
input_size = 1  # 입력 특성의 수
hidden_size = 20  # hidden state의 크기
output_size = 1  # 출력의 크기
model = MyModel(input_size, hidden_size, output_size)

device = torch.device("mps" if torch.mps.is_available() else "cpu")
model.to(device)

# 옵티마이저 설정
optimizer = Adam(model.parameters(), lr=0.001)

# 손실 함수 정의
criterion = nn.BCELoss()

MyModel(
  (fc): Linear(in_features=20, out_features=1, bias=True)
  (sigmoid): Sigmoid()
  (gru_cell): MyGRUCell(
    (x2h): Linear(in_features=1, out_features=60, bias=True)
    (h2h): Linear(in_features=20, out_features=60, bias=True)
  )
)

### 학습 부분

In [74]:


from tqdm.notebook import tqdm

num_epochs = 3

for epoch in tqdm(range(num_epochs), desc="Epochs"):
    model.train()
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}", leave=False):
        input_ids = batch['input_ids'].unsqueeze(dim=-1).to(device)
        labels = batch['labels'].float().to(device)
        optimizer.zero_grad()
        outputs = model(input_ids)[0]
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
    
    # 평가
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch['input_ids'].unsqueeze(dim = -1).to(device)
            labels = batch['labels'].float().to(device)
            
            outputs = model(input_ids)[0]
            predicted = (outputs.squeeze() > 0.5).long()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Test Accuracy: {accuracy:.4f}")

Epochs:   0%|          | 0/3 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import time

In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
.progress-bar-wrapper {
    background-color: #1f242b !important;
}
</style>
"""))


In [9]:
from tqdm.notebook import tqdm
import time

num_epochs = 3
num_batches = 2

for i in tqdm(range(num_epochs), desc="Epochs", position=0):
    for j in tqdm(range(num_batches), desc="Batches", leave=False):
        time.sleep(0.5)


Epochs:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [82]:
from tqdm import tqdm
import time

outer = tqdm(range(5), desc="Outer Loop")
for i in outer:
    inner = tqdm(range(10), desc="Inner Loop", leave=False)
    for j in inner:
        time.sleep(0.1)  # 작업 대체
        inner.update()


Outer Loop:   0%|          | 0/5 [00:00<?, ?it/s]







Outer Loop:  20%|██        | 1/5 [00:01<00:04,  1.06s/it]







Outer Loop:  40%|████      | 2/5 [00:02<00:03,  1.06s/it]







Outer Loop:  60%|██████    | 3/5 [00:03<00:02,  1.06s/it]







Outer Loop:  80%|████████  | 4/5 [00:04<00:01,  1.06s/it]







Outer Loop: 100%|██████████| 5/5 [00:05<00:00,  1.06s/it]


In [68]:
outputs

(tensor([[0.5969],
         [0.5969],
         [0.5969],
         [0.2475],
         [0.5969],
         [0.5969],
         [0.5969],
         [0.5969]], device='mps:0', grad_fn=<SigmoidBackward0>),
 tensor([[ 1.3186e-01,  6.6151e-01,  1.7024e-01, -6.5320e-01, -2.5124e-02,
          -6.9553e-01,  1.5358e-01,  2.1845e-01,  5.8105e-01,  7.3270e-01,
          -7.5735e-01,  1.3471e-01,  1.1159e-01,  1.7309e-01,  5.1598e-01,
           1.6811e-01, -1.8306e-01,  1.2173e-01,  9.0791e-02,  3.2315e-01],
         [ 1.3186e-01,  6.6151e-01,  1.7024e-01, -6.5320e-01, -2.5124e-02,
          -6.9553e-01,  1.5358e-01,  2.1845e-01,  5.8105e-01,  7.3270e-01,
          -7.5735e-01,  1.3471e-01,  1.1159e-01,  1.7309e-01,  5.1598e-01,
           1.6811e-01, -1.8306e-01,  1.2173e-01,  9.0791e-02,  3.2315e-01],
         [ 1.3186e-01,  6.6151e-01,  1.7024e-01, -6.5320e-01, -2.5124e-02,
          -6.9553e-01,  1.5358e-01,  2.1845e-01,  5.8105e-01,  7.3270e-01,
          -7.5735e-01,  1.3471e-01,  1.1159e-01,  

---
## 조금 돌아가자

In [54]:
class MyRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(MyRNN, self).__init__()
        self.hidden_size = hidden_size
        self.input_to_hidden = nn.Linear(input_size + hidden_size, hidden_size)
        self.activation = nn.Tanh()

    def forward(self, x, hidden):
        combined = torch.cat((x, hidden), dim=1)
        print(combined)
        hidden = self.activation(self.input_to_hidden(combined))
        return hidden

In [55]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = MyRNN(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # Initialize hidden state with zeros
        hidden = torch.zeros(x.size(0), self.hidden_size).to(x.device)
        
        # Forward propagate RNN
        for i in range(x.size(1)):
            hidden = self.rnn(x[:, i, :], hidden)
        
        # Decode the hidden state of the last time step
        out = self.fc(hidden)
        return out

In [56]:
# Example usage:
input_size = 10
hidden_size = 20
output_size = 5
seq_length = 15
batch_size = 32

# Create an instance of the model
model = SimpleRNN(input_size, hidden_size, output_size)

# Create a random input tensor
x = torch.randn(batch_size, seq_length, input_size)

# Forward pass
output = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

tensor([[ 0.8899,  0.0105,  1.1043, -1.4649, -0.9151, -0.7541,  0.7091, -0.0788,
          1.1716, -0.9525,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 1.1197,  1.9540, -0.0753, -1.1652,  1.7873,  0.3529, -0.2902, -0.8414,
         -0.0315, -1.0678,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.1074,  1.1943, -0.9353, -0.5472, -0.8126, -0.9496, -0.3192, -0.2257,
         -2.6229, -0.3460,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.9811, -1.5744, -1.3089, -0.8569,  0.3009,  0.5747, -0.2040,  1.0021